<div class="alert alert-block alert-info">
    <b><h1>CE 3372 Water Systems Design</h1></b> 
</div>

# EPANET Project Workshop

:::{admonition} Course Website
[http://54.243.252.9/ce-3372-webroot/](http://54.243.252.9/ce-3372-webroot/)
:::

## Pecos Raw Water Transmission 

Overview here

### Problem Statement Review



:::{tip}
This section in progress
:::

## Supplied Data

[Link to Supplied Data Files](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/2-SuppliedData)

## Base Maps

Base maps are constructed using QGIS (ArcGIS would be fine too), and the given .kml file.

The figure below is an annotated map of the model area.  The three main parts are shown as the Terminal Storage Portion, the Transmission Pipeline Portion, and the Wellfield Portion (Collection is used interchangeably). 

![](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/5-Figures/overview.png)

The unlabeled contour interval is 100 feet. The approximate elevation of the wellfield is 2800 feet, and the terminal storage area is approximately 2600 feet. 

The KML file that contained the coordinates of the wellfield components, pipeline alignment, and terminal storage were loaded into QGIS to build an EPANET network.  A 30-meter DEM was downloaded from the public STRM database.  The STRM data were re-projected onto the Zone 31N UTM coordinate system (approximately Cartesian at the study scale; it will render nicely in ordinary EPANET, and the auto-length algorithm can determine pipe lengths from node locations).

:::{note}
The Shuttle Radar Topography Mission (SRTM) was flown aboard the space shuttle Endeavour February 11-22, 2000. The National Aeronautics and Space Administration (NASA) and the National Geospatial-Intelligence Agency (NGA) participated in an international project to acquire radar data which were used to create the first near-global set of land elevations.  Endeavour orbited Earth 16 times each day during the 11-day mission, completing 176 orbits. SRTM successfully collected single-pass interferometry radar data over 80% of the Earth's land surface between 60° north and 56° south lati-tude with data points posted every 1 arc-second (approximately 30 meters).
:::

A “plug-in” named “QEPANET” was used in QGIS to map nodes and tanks, and extract locations and elevations.  The extracted file is further edited by-hand until the remainder of the EPANET model is built, to be used directly in the US EPA supplied software.  

:::{warning}
QEPANET (circa 2024) did not work completely with QGIS - it could run an existing model withn QGIS, but additions and changes to the model would not run in QGIS; however it does work well enough to extract spatial coordinates and elevations for “node-type” objects, so it was used herein to obtain coordinates and elevations for use in the EPANET model.  Visualize its use as a step before the EPANET network map step, where a framework is built in GIS, then exported to an EPANET input file, then that file is loaded into EPANET to complete the model build.
:::

The resulting "model building basemap" for the class is (the image itself is a link to the file):

[![Pecos Pipeline Base Map](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/4-BaseMaps/PecosPipelineBasemapVertical.bmp)](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/4-BaseMaps/PecosPipelineBasemapVertical.bmp)


## Illustrative Examples

The goal is to design operation rules to maintain prescribed system conditions under varying demands.  Main goals are positive pressure for entire range of flows, head should be at least 3 feet - to guarentee that head in pipeline is at least 1.5 feet above the top of the pipe.   Maximum pressure is obtained from bursting data for the material:

[![](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/5-Figures/burstpressure.png)](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/5-Figures/burstpressure.png)

So run analysis 1X to identify if system works without draining tanks, heads are all greater than node values by required amount.  Then again with varying demand to see if still true.  Check for maximum pressures.

### No control rules

The link(s) to files for no-control case (just vary demands )

[ppss-wells-pipeline-control.net](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/8-EN2-Files/ppss-wells-pipeline-control.net)



## Tanks

In EPANET, when a tank is configured to allow overflow, any water that exceeds the tank’s maximum level is simply removed from the hydraulic system. In other words, the excess water is treated as a loss from the network, not routed to another node or storage element.

What happens internally:
- When the computed tank level would exceed the specified maximum water level:
- The tank level is capped at the maximum level.
- Any additional inflow that would raise the level further is treated as overflow discharge.
- That overflow volume is removed from the system mass balance.

Consequence for volume balance:

- Because the overflow water is not tracked to a receiving node, the system water balance will show a mismatch if you compare:

Inflow − Outflow ≠ ΔStorage

The difference corresponds to the overflow loss.

Physical interpretation

EPANET assumes the water simply spills out of the tank through an overflow pipe to some external drainage location, such as: a storm drain, a drainage ditch, ground surface, an overflow pipe not modeled in the network

But the destination is not simulated.

Where you can see it

Overflow is typically reported in the tank results table as:
- Overflow rate
- Total overflow volume (depending on the interface you are using).

Practical modeling notes:

- Many engineers use overflow intentionally to detect operational problems:
- Pumps oversized or improperly controlled
- Tank control levels set incorrectly
- Demand patterns causing tank overfill

For some examples 
- If pumps continue running after a tank is full, the excess flow simply leaves the system as overflow.
- This is often a cue that pump controls should be tied to tank level.

                    Q_in
                     ↓
              ┌─────────────┐
              │    TANK     │
              │             │
              │   storage   │
              └─────────────┘
                 ↓       ↓
             Q_out   Q_overflow
                      (lost from
                      the model)

In EPANET, when a tank is set with Overflow = No, the tank behaves as a closed storage element at its maximum level.

What happens when the tank reaches maximum level
- If inflow would raise the tank above its Maximum Water Level, EPANET enforces the limit by preventing further inflow.

The solver handles this by:
- Fixing the tank head at the elevation corresponding to the maximum water level.

- The tank then behaves like a fixed-head node (reservoir) at that head.

- Any pipes or pumps connected to the tank adjust their flows accordingly.

Hydraulic consequences

- Inflow to the tank becomes zero once the tank is full.

- Pumps feeding the tank may:
  - operate at zero flow, or
  - shut off if controls or pump curves require it.
  - Flow may reverse through connecting pipes if system pressure exceeds the tank head.

Physical interpretation

This setting represents tanks where:
- automatic pump controls prevent overfilling
- level sensors shut down pumps
- float valves or control valves stop inflow

Practical modeling note

In most water distribution operational models, engineers use Overflow = No, because real systems typically include controls to prevent continuous spilling.

The Overflow = Yes option is usually used when:

- modeling old standpipes or open reservoirs
- checking system resilience during control failures
- evaluating worst-case pump operation

                    Q_in
                     ↓
              ┌─────────────┐
              │    TANK     │
              │             │
              │   storage   │
              └─────────────┘
                     ↓
                   Q_out

### Tank Control File(s)

[Simple File for Exploring Control Rules](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/8-EN2-Files/tank-control-example.net)

### Control rules

The link(s) to files for active controls 

[ppss-wells-pipeline-control.net](http://54.243.252.9/ce-3372-webroot/2-Exercises/PR1_PecosWater/8-EN2-Files/ppss-control-case6A.net)